# Fine tune SBERT 

In [1]:
import pandas as pd 

import pickle
import json
import os

from collections import Counter
from itertools import combinations 

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sentence_transformers import InputExample, SentenceTransformer, losses, SentencesDataset, LoggingHandler, evaluation
from torch.utils.data import DataLoader

import logging

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import requests
import pickle
from io import BytesIO

record_id = "15576145"
access_token = "AUn57Ykztzzg7Cy8MdtAFEzGc53A4StqkVAYehmIa8lXe03evN1rHvV7GwZ6"  # Put your token here

api_url = f"https://zenodo.org/api/records/{record_id}"
headers = {"Authorization": f"Bearer {access_token}"}

# Get metadata
response = requests.get(api_url, headers=headers)
response.raise_for_status()
metadata = response.json()

# Find .pkl file and get the 'self' link
file_url = None
filename = None
for file in metadata.get('files', []):
    if file['key'].endswith('.pkl'):
        filename = file['key']
        file_url = file['links']['self']
        break

if not file_url:
    raise FileNotFoundError("No .pkl file found in the record.")

print(f"Downloading {filename} from {file_url}")

# Download file with auth header
file_response = requests.get(file_url, headers=headers)
file_response.raise_for_status()

# Load pickle from bytes
data = pickle.load(BytesIO(file_response.content))

print("Pickle file loaded:", type(data))


Pickle file loaded: <class 'list'>


In [4]:
# select just 10% of train_examples
train_examples = data[:int(len(data) * 0.1)]

In [5]:
# summary stats of train_examples
print(f"Number of training examples: {len(train_examples)}")

Number of training examples: 890988


In [6]:
# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s',
                    level=logging.INFO)

In [7]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=32,
    num_workers=4  # Adjust based on your CPU
)

/opt/conda/envs/nlp_env_analysis/lib/python3.10/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 1, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model=model)

2025-06-02 14:29:00,629 - Use pytorch device_name: cpu
2025-06-02 14:29:00,630 - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [10]:
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=2)

KeyboardInterrupt: 

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path='output/my_model',
    show_progress_bar=True
)

ValueError: You have set `args.eval_strategy` to steps, but you didn't provide an `eval_dataset` or an `evaluator`. Either provide an `eval_dataset` or an `evaluator` to `SentenceTransformerTrainer`, or set `args.eval_strategy='no'` to skip evaluation.